### Cell 1: Imports

This cell imports all the necessary libraries for the project.

In [1]:
#!/usr/bin/env python3
import os
import json
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    roc_curve,
    precision_recall_curve
)
import joblib

# Optional SHAP
try:
    import shap
    print("[INFO] SHAP library found.")
except ImportError:
    shap = None
    print("[INFO] SHAP library not found, skipping SHAP-related features.")

print("All libraries imported successfully.")

C:\Users\User\anaconda3\envs\Project\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[INFO] SHAP library found.
All libraries imported successfully.


### Cell 2: Configuration

**⬇️ ⬇️ Update your dataset name here.** This cell also defines the constants, feature names, and protocol mappings your script uses.

In [2]:
# ===============================================================
# CONFIGURATION
# ===============================================================


csv_path = "CICIDS2017_subset.csv"  

# --- Feature and Label Definitions ---
PROTO_MAP = {'tcp': 0, 'udp': 1, 'icmp': 2, 'others': 3}

FEATURE_KEYS = [
    'Protocol', 'Flow Duration', 'Total Fwd Packets', 'Total Backward Packets',
    'Total Length of Fwd Packets', 'Total Length of Bwd Packets',
    'Fwd Packet Length Mean', 'Bwd Packet Length Mean',
    'Flow IAT Mean', 'Fwd IAT Mean', 'Bwd IAT Mean',
    'Fwd PSH Flags', 'Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags'
]

LABEL_COL = 'Label'

print(f"Dataset path set to: {csv_path}")
print(f"Tracking {len(FEATURE_KEYS)} features.")

Dataset path set to: CICIDS2017_subset.csv
Tracking 15 features.


### Cell 3: Helper Functions

This cell defines the `encode_protocol` and `preprocess_dataframe` functions that will be used to clean and prepare the data.

In [3]:
# ===============================================================
# HELPER FUNCTIONS
# ===============================================================

def encode_protocol(proto_value):
    """Converts protocol string/number to a standard float."""
    if pd.isna(proto_value):
        return 3.0
    val = str(proto_value).lower()
    return float(PROTO_MAP.get(val, 3))

def preprocess_dataframe(df):
    """Cleans, encodes, and prepares the DataFrame for training."""
    
    print(f"[INFO] Original columns: {len(df.columns)}")
    df.columns = df.columns.str.strip()
    df.replace([np.inf, -np.inf], np.nan, inplace=True)

    missing_cols = [col for col in FEATURE_KEYS if col not in df.columns]
    if missing_cols:
        print(f"[WARN] Missing required feature columns: {missing_cols}")
        print("[WARN] Filling missing feature columns with 0.0")
        for col in missing_cols:
            df[col] = 0.0

    if LABEL_COL not in df.columns:
        raise ValueError(f"Label column '{LABEL_COL}' not found in CSV.")

    df[FEATURE_KEYS] = df[FEATURE_KEYS].fillna(0.0)

    # Encode protocol if string/object
    if 'Protocol' in df.columns:
        print("[INFO] Encoding 'Protocol' column...")
        if df['Protocol'].dtype == 'object' or pd.api.types.is_string_dtype(df['Protocol']):
            df['Protocol'] = df['Protocol'].apply(encode_protocol)
        else:
            # Assumes numeric mapping (e.g., 6=TCP, 17=UDP, 1=ICMP)
            df['Protocol'] = df['Protocol'].map({6:0, 17:1, 1:2}).fillna(3)

    print("[INFO] Encoding 'Label' column (Benign=0, Attack=1)...")
    y = df[LABEL_COL].apply(lambda x: 0 if str(x).strip().upper() in ('BENIGN','NORMAL','0') else 1)

    X = df[FEATURE_KEYS].astype(np.float64)
    print(f"[INFO] Preprocessing complete. X shape: {X.shape}, y labels: {len(y)}")
    return X, y

print("Helper functions defined.")

Helper functions defined.


### Cell 4: Load and Preprocess Data

This cell executes the data loading and preprocessing steps using the `csv_path` from Cell 2 and the functions from Cell 3.

In [4]:
# ===============================================================
# LOAD AND PREPROCESS DATA
# ===============================================================
print(f"[INFO] Loading CSV: {csv_path}")
try:
    df = pd.read_csv(csv_path)
    print(f"Successfully loaded. Shape: {df.shape}")
    X, y = preprocess_dataframe(df)
    
    # Clean up memory
    del df
    
except FileNotFoundError:
    print(f"[ERROR] File not found at: {csv_path}")
    print("[ERROR] Please update 'csv_path' in Cell 2.")
except Exception as e:
    print(f"[ERROR] Could not read or process CSV: {e}")

[INFO] Loading CSV: CICIDS2017_subset.csv
Successfully loaded. Shape: (225745, 79)
[INFO] Original columns: 79
[WARN] Missing required feature columns: ['Protocol']
[WARN] Filling missing feature columns with 0.0
[INFO] Encoding 'Protocol' column...
[INFO] Encoding 'Label' column (Benign=0, Attack=1)...
[INFO] Preprocessing complete. X shape: (225745, 15), y labels: 225745


### Cell 5: Split Data for Training and Testing

Here, we split the processed data into training and testing sets.

In [5]:
# ===============================================================
# TRAIN-TEST SPLIT
# ===============================================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)
print(f"[INFO] Data split complete.")
print(f"  Total samples: {len(y)}")
print(f"  Training samples: {len(y_train)}")
print(f"  Testing samples:  {len(y_test)}")

[INFO] Data split complete.
  Total samples: 225745
  Training samples: 158021
  Testing samples:  67724


### Cell 6: Scale Data and Train Models

This cell scales the features and trains both the `RandomForestClassifier` (supervised) and `IsolationForest` (unsupervised) models.

In [6]:
# ===========================================================
# SCALE DATA AND TRAIN MODELS
# ===========================================================

# 1. Scale Data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print("[INFO] Data scaled using StandardScaler.")

# 2. Train RandomForestClassifier
print("[INFO] Training RandomForestClassifier...")
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train_scaled, y_train)
print("[INFO] RandomForestClassifier training complete.")

# 3. Train IsolationForest
print("[INFO] Training IsolationForest (unsupervised)...")
iso = IsolationForest(n_estimators=100, contamination=0.1, random_state=42, n_jobs=-1)
iso.fit(X_train_scaled)
print("[INFO] IsolationForest training complete.")

[INFO] Data scaled using StandardScaler.
[INFO] Training RandomForestClassifier...
[INFO] RandomForestClassifier training complete.
[INFO] Training IsolationForest (unsupervised)...
[INFO] IsolationForest training complete.


### Cell 7: (Optional) Initialize SHAP Explainer

If you have the `shap` library installed, this cell creates an explainer object to analyze feature importance later.

In [7]:
# ===========================================================
# SHAP EXPLAINER (Optional)
# ===========================================================
explainer = None
if shap is not None:
    try:
        print("[INFO] Creating SHAP TreeExplainer...")
        explainer = shap.TreeExplainer(rf)
        print("[INFO] SHAP explainer created successfully.")
    except Exception as e:
        print(f"[WARN] Could not create SHAP explainer: {e}")
else:
    print("[INFO] SHAP not installed — skipping feature explainability.")

[INFO] Creating SHAP TreeExplainer...
[INFO] SHAP explainer created successfully.


### Cell 8: Evaluate Model Performance

This cell runs the trained `RandomForest` model on the test set and prints a detailed evaluation, including accuracy, F1-score, and a classification report.

In [8]:
# ===========================================================
# EVALUATION
# ===========================================================
print("[INFO] Evaluating model on test data...")
y_pred = rf.predict(X_test_scaled)

try:
    y_score = rf.predict_proba(X_test_scaled)[:, 1]
except Exception:
    y_score = None
    print("[WARN] Could not get predict_proba scores.")

# Calculate metrics
acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, zero_division=0)
rec = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)
cm_arr = confusion_matrix(y_test, y_pred).tolist()

# Print results
print(f"\n[RESULT] Accuracy: {acc * 100:.2f}%")
print(f"[RESULT] Precision: {prec:.4f}")
print(f"[RESULT] Recall: {rec:.4f}")
print(f"[RESULT] F1 Score: {f1:.4f}")

print("\n[RESULT] Confusion Matrix:")
print(pd.DataFrame(cm_arr, index=["Actual Benign", "Actual Attack"], columns=["Pred. Benign", "Pred. Attack"]))

print("\n[RESULT] Classification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

[INFO] Evaluating model on test data...

[RESULT] Accuracy: 99.95%
[RESULT] Precision: 0.9997
[RESULT] Recall: 0.9994
[RESULT] F1 Score: 0.9995

[RESULT] Confusion Matrix:
               Pred. Benign  Pred. Attack
Actual Benign         29304            12
Actual Attack            24         38384

[RESULT] Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     29316
           1       1.00      1.00      1.00     38408

    accuracy                           1.00     67724
   macro avg       1.00      1.00      1.00     67724
weighted avg       1.00      1.00      1.00     67724



### Cell 9: Calculate Curve Data (ROC & PR)

This cell calculates the data points needed to plot ROC (Receiver Operating Characteristic) and PR (Precision-Recall) curves.

In [9]:
# ===========================================================
# CURVE DATA
# ===========================================================
roc_data, pr_data = {}, {}
if y_score is not None:
    try:
        fpr, tpr, _ = roc_curve(y_test, y_score)
        precision_curve, recall_curve, _ = precision_recall_curve(y_test, y_score)
        
        roc_data = {"fpr": fpr.tolist(), "tpr": tpr.tolist()}
        pr_data = {"precision": precision_curve.tolist(), "recall": recall_curve.tolist()}
        print("[INFO] ROC and PR curve data calculated.")
    except Exception as e:
        print(f"[WARN] Could not compute ROC/PR data: {e}")
else:
    print("[INFO] Skipping ROC/PR calculation (no probability scores).")

[INFO] ROC and PR curve data calculated.


### Cell 10: Save Models and Scaler

This cell saves the trained models (`RandomForest`, `IsolationForest`), the `StandardScaler`, and the `SHAP` explainer (if created) to the `models/` directory.

In [10]:
# ===========================================================
# SAVE MODELS
# ===========================================================
output_dir = "models"
os.makedirs(output_dir, exist_ok=True)

try:
    joblib.dump(scaler, os.path.join(output_dir, "scaler.joblib"))
    joblib.dump(rf, os.path.join(output_dir, "supervised_rf.joblib"))
    joblib.dump(iso, os.path.join(output_dir, "unsupervised_iso.joblib"))
    
    if explainer:
        joblib.dump(explainer, os.path.join(output_dir, "shap_explainer.joblib"))
    
    print(f"[INFO] All models and scaler saved to '{output_dir}/' directory.")

except Exception as e:
    print(f"[ERROR] Could not save models: {e}")

[INFO] All models and scaler saved to 'models/' directory.


### Cell 11: Extract and Display Feature Importance

This cell extracts the feature importances from the `RandomForest` model and prints the top 10 most influential features.

In [11]:
# ===========================================================
# FEATURE IMPORTANCE EXTRACTION
# ===========================================================
top_features = []
try:
    importances = rf.feature_importances_
    feat_imp = sorted(
        zip(FEATURE_KEYS, importances),
        key=lambda x: x[1],
        reverse=True
    )[:10]
    
    top_features = [{"feature": f, "importance": float(imp)} for f, imp in feat_imp]

    print("\n[INFO] Top 10 Important Features:")
    print("-" * 40)
    print(f"{'Feature':<30s}: {'Importance'}")
    print("-" * 40)
    for f, imp in feat_imp:
        print(f"  {f:<30s}: {imp:.5f}")
        
except Exception as e:
    print(f"[WARN] Could not extract feature importances: {e}")


[INFO] Top 10 Important Features:
----------------------------------------
Feature                       : Importance
----------------------------------------
  Fwd Packet Length Mean        : 0.25624
  Total Length of Fwd Packets   : 0.14633
  Total Fwd Packets             : 0.14594
  Fwd IAT Mean                  : 0.13753
  Bwd Packet Length Mean        : 0.08093
  Total Backward Packets        : 0.06712
  Total Length of Bwd Packets   : 0.05729
  Bwd IAT Mean                  : 0.04722
  Flow Duration                 : 0.03202
  Flow IAT Mean                 : 0.02854


### Cell 12: Save Final Metrics to JSON

Finally, this cell gathers all the evaluation metrics into a dictionary and saves them as a JSON file in the `static/data/` directory.

In [12]:
# ===========================================================
# METRICS JSON OUTPUT
# ===========================================================

metrics = {
    "accuracy": float(acc),
    "precision": float(prec),
    "recall": float(rec),
    "f1": float(f1),
    "confusion_matrix": cm_arr,
    "roc_data": roc_data,
    "pr_data": pr_data,
    "top_features": top_features
}

out_dir = os.path.join("static", "data")
os.makedirs(out_dir, exist_ok=True)
out_path = os.path.join(out_dir, "metrics.json")

try:
    with open(out_path, "w") as fh:
        # Saving 'basic_metrics' key for compatibility with original script
        json.dump({"basic_metrics": metrics, **metrics}, fh, indent=2)
    print(f"\n[INFO] All metrics written to {out_path}")
except Exception as e:
    print(f"[ERROR] Could not write metrics JSON: {e}")

print("\n--- [SCRIPT COMPLETE] ---")


[INFO] All metrics written to static\data\metrics.json

--- [SCRIPT COMPLETE] ---
